<a href="https://colab.research.google.com/github/NattInsights/ds/blob/main/tennis_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving atp_tennis.csv to atp_tennis.csv


In [ ]:
df = pd.read_csv('atp_tennis.csv', sep=',')

In [ ]:
df.head()

,Tournament,Date,Series,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2,Score
0,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Arthurs W.,Gambill J.M.,Gambill J.M.,105,58,-1,-1,-1.0,-1.0,6-3 6-7 4-6
1,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Balcells J.,Henman T.,Henman T.,218,11,-1,-1,-1.0,-1.0,4-6 6-7
2,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Clement A.,Enqvist T.,Enqvist T.,56,5,-1,-1,-1.0,-1.0,3-6 3-6
3,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Dosedel S.,Ljubicic I.,Dosedel S.,63,77,-1,-1,-1.0,-1.0,6-4 6-2
4,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Draper S.,Norman M.,Norman M.,155,15,-1,-1,-1.0,-1.0,4-6 4-6


In [ ]:
df.describe()

,Best of,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2
count,68635.000000,68635.000000,68635.000000,68635.000000,68635.000000,68635.000000,68635.000000
mean,3.377213,75.863087,75.422452,1144.517841,1151.747010,2.422493,2.414463
std,0.782397,100.242693,100.652846,1728.297181,1747.850943,2.677432,2.639093
min,3.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,3.000000,25.000000,24.000000,159.000000,161.000000,1.360000,1.353292
50%,3.000000,54.000000,54.000000,709.000000,710.000000,1.773750,1.800000
75%,3.000000,92.000000,92.000000,1258.000000,1265.000000,2.750000,2.750000
max,5.000000,3390.000000,4915.000000,16950.000000,16950.000000,67.000000,51.000000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68635 entries, 0 to 68634
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Tournament  68635 non-null  object 
 1   Date        68635 non-null  object 
 2   Series      68635 non-null  object 
 3   Court       68635 non-null  object 
 4   Surface     68635 non-null  object 
 5   Round       68635 non-null  object 
 6   Best of     68635 non-null  int64  
 7   Player_1    68635 non-null  object 
 8   Player_2    68635 non-null  object 
 9   Winner      68635 non-null  object 
 10  Rank_1      68635 non-null  int64  
 11  Rank_2      68635 non-null  int64  
 12  Pts_1       68635 non-null  int64  
 13  Pts_2       68635 non-null  int64  
 14  Odd_1       68635 non-null  float64
 15  Odd_2       68635 non-null  float64
 16  Score       68635 non-null  object 
dtypes: float64(2), int64(5), object(10)
memory usage: 8.9+ MB


In [ ]:
# before feature construction, check data quality for unusual cases
# like retirements, walkovers or nulls
print(df['Score'].isna().sum())

0


In [ ]:
print(df['Score'].value_counts().head())

Score
6-3 6-4    1264
3-6 4-6    1236
4-6 4-6    1229
6-4 6-4    1201
3-6 2-6     815
Name: count, dtype: int64


Set attribute feature construction testing logic on limited data

In [ ]:
sets = df['Score'].str.split(' ')
print(sets.head())

0    [6-3, 6-7, 4-6]
1         [4-6, 6-7]
2         [3-6, 3-6]
3         [6-4, 6-2]
4         [4-6, 4-6]
Name: Score, dtype: object


In [ ]:
score = '6-2'
games = score.split('-')
print(games)

['6', '2']


In [ ]:
games = np.array(games, dtype=int)
print(games)

[6 2]


In [ ]:
# 1. Define a function to calculate sets won by each player
def calculate_sets(score):

    # Handle missing or invalid scores
    if pd.isna(score) or not isinstance(score, str):
        return np.nan

    set_scores = score.split(" ")

    p1_sets = 0
    p2_sets = 0

    # 2. Process each set
    for s in set_scores:

        games = np.array(s.split("-"), dtype=int)

        # 3. Compare the games won by each player
        if games[0] > games[1]:
            p1_sets += 1
        elif games[1] > games[0]:
            p2_sets += 1

    # 4. Return the final set score
    return f"{p1_sets}-{p2_sets}"

# 5. Apply the function to the entire Score column
df["Sets"] = df["Score"].apply(calculate_sets)

# 6. Inspect the results
print(df[["Score", "Winner", "Sets"]].head(10))

         Score        Winner Sets
0  6-3 6-7 4-6  Gambill J.M.  1-2
1      4-6 6-7     Henman T.  0-2
2      3-6 3-6    Enqvist T.  0-2
3      6-4 6-2    Dosedel S.  2-0
4      4-6 4-6     Norman M.  0-2
5  6-7 7-5 6-3     Escude N.  2-1
6  7-6 5-7 6-4   Fromberg R.  2-1
7  6-3 3-6 4-6      Vinck C.  1-2
8      6-2 6-1   Grosjean S.  2-0
9  6-3 2-6 6-1     Hewitt L.  2-1
